In [ ]:
from clodius.core import Tileset
from clodius.tiles_v2.bigwig import BigWigTileset
from clodius.tiles_v2.bigbed import BigBedTileset
from clodius.tiles_v2.cooler import CoolerTileset
from clodius.tiles_v2.multivec import MultivecTileset
from clodius.tiles_v2.bed import BedTileset
from clodius.core import Chromsizes, TileId
from higlass.tilesets import ClodiusTileset
import higlass as hg
import bioframe

hg38 = bioframe.assembly_info("hg38")


def chromsizes_tileset(assembly: str) -> ClodiusTileset:
    assembly = bioframe.assembly_info(assembly)
    chromsizes = Chromsizes.from_series(assembly.chromsizes)
    return ClodiusTileset(
        "vector",
        tiles_impl=lambda _: [],  # does not actually serve tiles
        info_impl=lambda: {
            "max_width": chromsizes.total_length,
            "chromsizes": chromsizes.to_pairs(),
            "min_pos": [0],
            "max_pos": [chromsizes.total_length],
        },
    )


def clodius_tileset(ts: Tileset) -> ClodiusTileset:

    def tiles_wrapper(tids_raw: list[str]) -> list[tuple[str, bytes]]:
        # higlass-python keys its response dict by whatever we return here, then
        # JSON-encodes it -- so the key must be the raw string, not a TileId.
        # TileId.raw exists for exactly this: the client matches responses to
        # requests by exact string.
        tids = [
            TileId.parse(
                raw, ndim=ts.ndim, modifiers=ts.modifiers, options=ts.options or None
            )
            for raw in tids_raw
        ]
        return [(tid.raw, payload) for tid, payload in ts.tiles(tids)]


    def info_wrapper():
        # exclude_none: pydantic emits every unset optional field as null, and
        # `null` is not interchangeable with *absent* on the wire. Hygiene --
        # tested innocent as a cause of the render failure, which was stale
        # widgets holding id()-derived uids that no longer resolve.
        return ts.info().model_dump(exclude_none=True)

    return ClodiusTileset(
        ts.datatype,
        tiles_impl=tiles_wrapper,
        info_impl=info_wrapper,
    )


In [ ]:
cs = chromsizes_tileset("hg38")
ts = BigWigTileset(
    "/Users/nezar/local/devel/higlass/clodius/_scratch/hlc_h3k27me3.bigwig",
    chromsizes=Chromsizes.from_series(hg38.chromsizes)
)
cts = clodius_tileset(ts)
view = hg.view(
    (cs.track("chromosome-labels"), "top"),
    (cts.track(), "top"),
    initialXDomain=(0, 1e6)
)
view

In [ ]:
cs = chromsizes_tileset("hg38")
ts = BigBedTileset(
    "/Users/nezar/local/devel/higlass/clodius/_scratch/hlc_h3k27me3.bigwig",
    chromsizes=Chromsizes.from_series(hg38.chromsizes)
)
cts = clodius_tileset(ts)
view = hg.view(
    (cs.track("chromosome-labels"), "top"),
    (cts.track(), "top"),
    initialXDomain=(0, 1e6)
)
view

In [ ]:
cs = chromsizes_tileset("hg38")
ts = BedTileset(
    "/Users/nezar/local/devel/higlass/clodius/_scratch/GRCh38-cCREs.bed.gz",
    chromsizes=Chromsizes.from_series(hg38.chromsizes),
    index_path="/Users/nezar/local/devel/higlass/clodius/_scratch/GRCh38-cCREs.bed.gz.tbi"
)
cts = clodius_tileset(ts)
view = hg.view(
    (cs.track("chromosome-labels"), "top"),
    (cts.track(), "top"),
    initialXDomain=(0, 1e6)
)
view

In [ ]:
cs = chromsizes_tileset("hg38")
cooler_ts = CoolerTileset(
    "/Users/nezar/local/devel/higlass/clodius/_scratch/H1_HB_HiC3_20240517.hg38.mapq_30.1000.mcool"
)
ccts = clodius_tileset(cooler_ts)
view = hg.view(
    (cs.track("chromosome-labels"), "top"),
    (ccts.track("heatmap"), "center"),
    (cs.track("2d-chromosome-grid", options={"lineStrokeWidth": 2}), "center"),
    initialXDomain=(1e6, 3e9),
    initialYDomain=(1e6, 3e9),
)
view

In [ ]:
cs = chromsizes_tileset("hg38")
mv = clodius_tileset(MultivecTileset(
    "/Users/nezar/local/devel/higlass/clodius/test/sample_data/chrm_boundaries_test.multires.mv5",
))
view = hg.view(
    (cs.track("chromosome-labels"), "top"),
    (cs.track("horizontal-chromosome-grid"), "top"),
    (mv.track("multivec"), "top"),
    initialXDomain=(1e6, 3e9),
)
view
